# Dynamics via Trotterization

Here, we show a simple example of how to simulate dynamics using Trotterization. We first need to create an electronic Hamiltonian from a molecular geometry. Here, we use $\mathrm{C_2H_6S_2}$ (dimethyl disulfide) as an example. 

In [1]:
import warnings
warnings.filterwarnings("ignore")

from rdkit import Chem
from rdkit.Chem import AllChem
import cupynumeric as np

import trovolut as trv

In [2]:
dmds_smiles = "CSSC"

# RDKit: SMILES -> mol
mol = Chem.MolFromSmiles(dmds_smiles)
mol = Chem.AddHs(mol)  # add hydrogens

# embed 3D coordinates
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

# extract XYZ as multiline string in standard XYZ format:
atoms = [atom.GetSymbol() for atom in mol.GetAtoms()]
conf = mol.GetConformer()

xyz_lines = []
xyz_lines.append(str(len(atoms)))
xyz_lines.append("dmds generated by RDKit")
for i, sym in enumerate(atoms):
    pos = conf.GetAtomPosition(i)
    xyz_lines.append(f"{sym} {pos.x:.8f} {pos.y:.8f} {pos.z:.8f}")

dmds_xyz = "\n".join(xyz_lines)

H_elec = trv.qm.ElectronicHamiltonian.from_xyz(
    xyz_source=dmds_xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
    n_core_keep=2,
    n_virt_keep=2,
).to_qubits()


converged SCF energy = -864.654938672564


We then get the terms of the Hamiltonian and decompose it for later use in Trotterization.

In [3]:
qmeta = H_elec.decompose()
hamcoeffs = H_elec.terms()

print("first 5 terms:")
for term in qmeta["terms"][:5]:
    print(term)

first 5 terms:
{'coeff': np.float64(152.29957877540937), 'pauli': 'I'}
{'coeff': np.float64(8.690552738296475), 'pauli': 'Z0'}
{'coeff': np.float64(9.72817262425162e-08), 'pauli': 'Y0Z1Z2Z3Y4'}
{'coeff': np.float64(9.72817262425162e-08), 'pauli': 'X0Z1Z2Z3X4'}
{'coeff': np.float64(-4.0259546478902515e-06), 'pauli': 'Y0Z1Z2Z3Z4Z5Y6'}


In [4]:
mats = [term['coef'] * term['matrix'] for term in hamcoeffs]

In [13]:
evol = np.linalg.expm(-1j * mats[0] * 0.01) 

psi0 = np.zeros(2**qmeta["n_qubits"], dtype=complex)
psi0[0] = 1.0  # initial state is |000...0>

dyn = evol @ psi0
dyn.shape

(256,)

In [8]:
from trovolut.qm.hamsim import simulate_hamiltonian_no_simulated_precision

In [9]:
psi0 = np.zeros(2**qmeta["n_qubits"], dtype=complex)
psi0[0] = 1.0  # initial state |000...0>
dyn = simulate_hamiltonian_no_simulated_precision(
    mats, psi0, 1, 10)

In [11]:
dyn[0].shape

(256,)